In [ ]:
from autumn.projects.sm_covid2.common_school.runner_tools import INCLUDED_COUNTRIES
import pandas as pd
from pathlib import Path
from autumn.projects.sm_covid2.common_school.output_plots import multicountry as mc
from matplotlib import pyplot as plt 
from importlib import reload

full_iso3_list = list(INCLUDED_COUNTRIES['all'].keys())

In [ ]:
analysis_folder = Path.cwd() / "33489767_test_full_analysis_24Jan2024_main"
diff_quantiles_dfs = {}
for iso3 in full_iso3_list:
    diff_quantiles_path = analysis_folder / iso3 / "diff_quantiles_df.parquet"
    if Path.exists(diff_quantiles_path):
        diff_quantiles_dfs[iso3] = pd.read_parquet(diff_quantiles_path)

In [ ]:
x_vars_labs = {
    "prop_kids": "% under 15 years old",
    "prop_elderly": "% above 70 years old",

    "stringency": "Average Oxford Stringency index during school closures",
    "prop_students": "% of population enrolled in schools",

    "n_weeks_closed": "n weeks schools closed"
}

import warnings
warnings.filterwarnings("ignore")

reload(mc)
req_outputs = ["cases_averted_relative", "deaths_averted_relative"]
heterogeneity_df = pd.DataFrame(columns=list(x_vars_labs.keys()) + req_outputs)
for iso3 in full_iso3_list:
    new_row = {x_var: mc.get_pop_characteristics(iso3, x_var) for x_var in x_vars_labs}
    new_row.update(
        {output: - 100. * diff_quantiles_dfs[iso3][output].loc[0.5] for output in req_outputs}
    )
    heterogeneity_df = pd.concat([heterogeneity_df, pd.DataFrame([new_row])], ignore_index=True)
heterogeneity_df = heterogeneity_df.dropna()

In [ ]:
import statsmodels.api as sm

def forward_selection(data, target, significance_level=0.5):
    initial_features = data.columns.tolist()[0:5]
    best_features = []
    while initial_features:
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features)
        for new_column in remaining_features:
            model = sm.OLS(target, sm.add_constant(data[best_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

# Example usage
# Assuming df is your dataframe and 'y' is the target variable
X = heterogeneity_df.drop(columns=['cases_averted_relative', 'deaths_averted_relative'])

output = 'deaths_averted_relative'
y = heterogeneity_df[output]

selected_features = forward_selection(X, y)
print(selected_features)
# selected_features = ['prop_students', 'prop_kids', 'prop_elderly', 'stringency', 'n_weeks_closed']
selected_features = ['prop_elderly', 'stringency', 'n_weeks_closed']


print("Selected features:", selected_features)

selected_X = X[selected_features]
model = sm.OLS(y, sm.add_constant(selected_X)).fit()

print(model.summary())

In [ ]:
import warnings
warnings.filterwarnings("ignore")

reload(mc)

hetero_path = Path.cwd() / "heterogeneity_plots"

for x_var, title in x_vars_labs.items():
    print(title)
    fig, axes = plt.subplots(2, 1, figsize=(10,10))
    for i, output in enumerate(["cases_averted_relative", "deaths_averted_relative"]): #, "delta_hospital_peak_relative"]:
        correlation_df = mc.make_icer_like_plot_generic(diff_quantiles_dfs, output, x_var=x_var, axis=axes[i], legend=[True, False][i])
        # corr = correlation_df['x_val'].corr(correlation_df["median_effect"])      

    for file_format in ["pdf"]:  
        fig.savefig(hetero_path / f"hetero_{x_var}.{file_format}", bbox_inches='tight')
    plt.close()